# P3 - Fig Engine real-path memory verification

Measures process RSS by phase for `FigModel.from_pretrained()` and Tier-1 training. The default is `lowram`, the mode relevant to the minimum-memory claim. Partial results are written atomically to Google Drive after every phase.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
P3_DIR = '/content/drive/MyDrive/littlefig-p3'
os.makedirs(P3_DIR, exist_ok=True)
print('Durable P3 directory:', P3_DIR)

In [ ]:
import os, subprocess
REPO = '/content/littlefig'
BRANCH = 'research/p1-figmezo-verify'
if not os.path.exists(REPO + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, 'https://github.com/Harboria-Labs/littlefig.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'switch', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'pull', '-q', '--ff-only'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', REPO], check=True)
os.chdir(REPO)
print('Source checkout ready:', BRANCH)

In [ ]:
MEMORY_MODE = 'lowram'  # lowram tests the minimum-memory claim; also try figcache or fast
STEPS = 20
BATCH_SIZE = 2
SEQUENCE_LENGTH = 256
RESULTS_PATH = P3_DIR + '/figengine_8gb_' + MEMORY_MODE + '_results.json'
cmd = [
    'python', 'benchmark/experiment_8gb_v1.py',
    '--memory-mode', MEMORY_MODE, '--steps', str(STEPS),
    '--batch-size', str(BATCH_SIZE), '--sequence-length', str(SEQUENCE_LENGTH),
    '--memory-budget-gib', '8.0', '--claim-memory-gib', '0.4',
    '--results-path', RESULTS_PATH,
]
print('Run:', ' '.join(cmd))

In [ ]:
subprocess.run(cmd, check=True)

In [ ]:
import json, os
print('Results:', RESULTS_PATH, os.path.exists(RESULTS_PATH))
if os.path.exists(RESULTS_PATH):
    print(json.dumps(json.load(open(RESULTS_PATH)), indent=2))